In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split, Dataset
from collections import defaultdict
import zipfile
from tqdm import tqdm
import shutil
import torchvision
import json
import pandas as pd
import time

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
OUTPUT_DIR = "/kaggle/working/experiment_results"
WEIGHTS_DIR = os.path.join(OUTPUT_DIR, "weights")
LOGS_DIR = os.path.join(OUTPUT_DIR, "logs")

os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

In [ ]:
def prepare_cats_dogs(input_zip_path, extract_to):

    if os.path.exists(extract_to):
        print("Dataset already extracted.")
        return

    print("Extracting Cats vs Dogs dataset")
    with zipfile.ZipFile(input_zip_path, 'r') as zip_ref:
        zip_ref.extractall(os.path.dirname(extract_to))
    
    source_dir = os.path.join(os.path.dirname(extract_to), 'train')
    target_dir = extract_to
    
    os.makedirs(os.path.join(target_dir, 'cat'), exist_ok=True)
    os.makedirs(os.path.join(target_dir, 'dog'), exist_ok=True)
    
    files = os.listdir(source_dir)
    for file in tqdm(files, desc="Organizing Images"):
        if file.startswith('cat'):
            shutil.move(os.path.join(source_dir, file), os.path.join(target_dir, 'cat', file))
        elif file.startswith('dog'):
            shutil.move(os.path.join(source_dir, file), os.path.join(target_dir, 'dog', file))

    os.rmdir(source_dir)

In [ ]:
KAGGLE_INPUT_ZIP = "/kaggle/input/dogs-vs-cats/train.zip"
PROCESSED_DATA_DIR = "/kaggle/working/data/cats_dogs_sorted"

has_cats_dogs = False
if os.path.exists(KAGGLE_INPUT_ZIP):
    prepare_cats_dogs(KAGGLE_INPUT_ZIP, PROCESSED_DATA_DIR)
    has_cats_dogs = True
else:
    print("'dogs-vs-cats' dataset not found in input. Skipping.")

## CNN architecture

In [ ]:

class CNN(nn.Module):
    def __init__(self, activation, init_type, num_classes):
        super().__init__()
        self.activation = activation

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)

        self.fc1 = nn.Linear(64 * 56 * 56, 256)
        self.fc2 = nn.Linear(256, num_classes)

        self.initialize_weights(init_type)

    def activate(self, x):
        if self.activation == "relu":
            return F.relu(x)
        elif self.activation == "tanh":
            return torch.tanh(x)
        elif self.activation == "leaky_relu":
            return F.leaky_relu(x)

    def initialize_weights(self, init_type):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                if init_type == "xavier":
                    nn.init.xavier_uniform_(m.weight)
                elif init_type == "kaiming":
                    nn.init.kaiming_uniform_(m.weight)
                else:
                    nn.init.normal_(m.weight, mean=0, std=0.01)

    def forward(self, x):
        x = self.pool(self.activate(self.bn1(self.conv1(x))))
        x = self.pool(self.activate(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout(self.activate(self.fc1(x)))
        return self.fc2(x)




## Data Loading

In [ ]:
def get_dataloaders(dataset_name, batch_size=64):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    if dataset_name == "cifar10":
         train_ds = datasets.CIFAR10(root="/kaggle/working/data", train=True,
                                     download=True, transform=transform)
         test_ds = datasets.CIFAR10(root="/kaggle/working/data", train=False,
                                    download=True, transform=transform)


         num_classes = 10

    elif dataset_name == "catsdogs":
        transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
        ])
        full_cd = torchvision.datasets.ImageFolder(root=PROCESSED_DATA_DIR, transform=transform)
        train_size = int(0.8 * len(full_cd))
        test_size = len(full_cd) - train_size
        train_ds, test_ds = random_split(full_cd, [train_size, test_size])
        num_classes=2
        

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader, num_classes

## Training and evaluation

In [ ]:
def train_and_evaluate(model, train_loader, test_loader, optimizer, epochs=40):
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs} started")
        model.train()
        for batch_idx, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            if batch_idx % 100 == 0:
                print(f"  Batch {batch_idx}, Loss: {loss.item():.4f}")

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total




## Grid search code for all different hyperparameters

In [ ]:
def grid_search(dataset_name):
    activations = ["relu", "tanh", "leaky_relu"]
    inits = ["xavier", "kaiming", "random"]
    optimizers = ["sgd", "adam", "rmsprop"]

    train_loader, test_loader, num_classes = get_dataloaders(dataset_name)

    best_acc = 0
    best_model = None
    best_config = None

    for act in activations:
        for init in inits:
            for opt_name in optimizers:

                model = CNN(act, init, num_classes).to(device)

                if opt_name == "sgd":
                    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
                elif opt_name == "adam":
                    optimizer = optim.Adam(model.parameters(), lr=0.001)
                else:
                    optimizer = optim.RMSprop(model.parameters(), lr=0.001)

                acc = train_and_evaluate(model, train_loader, test_loader, optimizer)

                print(f"{dataset_name} | {act} | {init} | {opt_name} -> {acc:.4f}")

                if acc > best_acc:
                    best_acc = acc
                    best_model = model
                    best_config = (act, init, opt_name)
                    best_act = act
                    best_init = init
                    best_opt = opt_name

    torch.save(best_model.state_dict(), f"{dataset_name}_{best_act}_{best_init}_{best_opt}.pth")
    print(f"\nBest CNN for {dataset_name}: {best_config}, Accuracy: {best_acc:.4f}\n")

    return best_acc

## Resnet18 Architecture

In [ ]:
def train_resnet18(dataset_name):
    train_loader, test_loader, num_classes = get_dataloaders(dataset_name)

    model = models.resnet18(pretrained=True)
    for param in model.parameters():
        param.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    model.to(device)

    optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    num_epochs = 40

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        print(f"\nEpoch [{epoch+1}/{num_epochs}]")

        for batch_idx, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            
            if (batch_idx) % 100 == 0:
                avg_loss = running_loss / (batch_idx + 1)
                print(
                    f"  Batch [{batch_idx+1}/{len(train_loader)}] "
                    f"Loss: {avg_loss:.4f}"
                )

        print(f"Epoch {epoch+1} finished. Avg Loss: {running_loss/len(train_loader):.4f}")

    
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    acc = correct / total
    torch.save(model.state_dict(), f"resnet18_{dataset_name}.pth")

    print(f"\nResNet-18 Accuracy on {dataset_name}: {acc:.4f}")
    return acc


In [ ]:
print("\n--- CIFAR-10 EXPERIMENT ---")
cnn_cifar = grid_search("cifar10")
resnet_cifar = train_resnet18("cifar10")

print("\n--- CATS vs DOGS EXPERIMENT ---")
cnn_catsdogs = grid_search("catsdogs")
resnet_catsdogs = train_resnet18("catsdogs")

print("\n--- FINAL COMPARISON ---")
print(f"CIFAR-10 | Best CNN: {cnn_cifar:.4f} | ResNet-18: {resnet_cifar:.4f}")
print(f"Cats vs Dogs | Best CNN: {cnn_catsdogs:.4f} | ResNet-18: {resnet_catsdogs:.4f}")